# Comparação do DemDS em C++ e Python

Salve este notebook em `notebooks/` e use o kernel do ambiente do projeto, com o pacote `complex_networks` instalado. A implementação Python é importada de `src/complex_networks/algorithms/mds/demds.py`.

Cada arquivo `.txt.gz` vira um `nx.Graph`. O notebook roda o executável C++ com uma instância temporária do mesmo grafo, roda o solver Python em memória e grava uma linha no CSV. `output_cpp` e `output_python` guardam o **tamanho** dos conjuntos dominantes. `time_python` mede em segundos a chamada do solver Python, depois de carregar e preparar o grafo.

## 1. Configurar a execução

Ajuste `CPP_BIN` para o executável original compilado. Cada execução do lote usa uma semente para todos os grafos. Para testar uma rede antes do lote inteiro, coloque o nome do arquivo em `GRAPH_PATTERN`.

In [16]:
import csv
import gc
import subprocess
from pathlib import Path
import sys
sys.path.append(str(Path.cwd().parent))

from tempfile import TemporaryDirectory
from time import perf_counter

import networkx as nx

from src.complex_networks.algorithms.mds.demds import mds_solver_demds
from src.complex_networks.config import PROJECT_ROOT, RAW_DATA_DIR, RESULTS_DIR, CPP_DIR
from src.complex_networks.io import load_edgelist, preprocess
from src.complex_networks.utils import iter_files


GRAPH_DIR = RAW_DATA_DIR
GRAPH_PATTERN = "*.txt.gz"
CPP_BIN = CPP_DIR / "DemDS" / "DemDS"
CSV_PATH = RESULTS_DIR / "benchmarks" / "mds" / "demds_results.csv"
FILE_ORDER = "size"

SEED = 1
TIME_LIMIT_SECONDS = 30
CPP_ALPHA_ARG = 90  # Quarto argumento da interface do executável original.
CPP_TIMEOUT_SECONDS = None  # Limite externo opcional para o processo C++.

print(f"Grafos em {GRAPH_DIR}")
print(f"Executável C++ em {CPP_BIN}")
print(f"Resultados em {CSV_PATH}")

Grafos em /home/fernando/complex-networks/data/raw
Executável C++ em /home/fernando/complex-networks/libs/DemDS/DemDS
Resultados em /home/fernando/complex-networks/results/benchmarks/mds/demds_results.csv


## 2. Definir a comparação de uma instância

A exportação para C++ usa os mesmos IDs internos da versão Python. A validação da solução Python ocorre depois da medição. O grafo fica em uma variável local e é liberado no bloco `finally`, mesmo quando uma execução falha.

In [9]:
def write_cpp_instance(G: nx.Graph, path: Path, n_edges: int) -> None:
    """Escreve o cabeçalho n m e uma aresta por linha para o DemDS C++."""
    if G.is_directed() or G.is_multigraph():
        raise ValueError("DemDS requer um grafo simples e não direcionado.")

    node_ids = {node: i for i, node in enumerate(G, start=1)}
    with path.open("w", encoding="ascii") as stream:
        stream.write(f"{len(node_ids)} {n_edges}\n")
        for u, v in G.edges():
            if u == v:
                raise ValueError("Remova os laços com preprocess antes da execução.")
            stream.write(f"{node_ids[u]} {node_ids[v]}\n")


def run_cpp(G: nx.Graph, executable: Path, n_edges: int) -> dict:
    with TemporaryDirectory(prefix="demds-") as temporary_dir:
        instance = Path(temporary_dir) / "instance.txt"
        write_cpp_instance(G, instance, n_edges)
        started = perf_counter()
        try:
            run = subprocess.run(
                [str(executable), instance.name, str(SEED),
                 str(TIME_LIMIT_SECONDS), str(CPP_ALPHA_ARG)],
                cwd=instance.parent,
                capture_output=True,
                text=True,
                encoding="utf-8",
                errors="replace",
                timeout=CPP_TIMEOUT_SECONDS,
            )
        except subprocess.TimeoutExpired:
            return {
                "status_cpp": "timeout",
                "time_cpp": perf_counter() - started,
                "error_cpp": f"Limite externo de {CPP_TIMEOUT_SECONDS} s.",
            }
        except OSError as error:
            return {
                "status_cpp": "error",
                "time_cpp": perf_counter() - started,
                "error_cpp": str(error),
            }
        elapsed = perf_counter() - started

    if run.returncode:
        return {
            "status_cpp": "error",
            "time_cpp": elapsed,
            "error_cpp": f"returncode={run.returncode} {run.stderr.strip()[-300:]}".strip(),
        }

    try:
        last_line = run.stdout.strip().splitlines()[-1]
        _, size_text, _ = last_line.rsplit(",", 2)
        size = int(size_text)
        if not 0 <= size <= G.number_of_nodes():
            raise ValueError("Cardinalidade fora do intervalo do grafo.")
    except (IndexError, ValueError) as error:
        return {
            "status_cpp": "invalid_output",
            "time_cpp": elapsed,
            "error_cpp": f"{error} Saída recebida {run.stdout.strip()[-300:]}",
        }

    return {"status_cpp": "ok", "output_cpp": size, "time_cpp": elapsed}


def run_python(G: nx.Graph) -> dict:
    started = perf_counter()
    try:
        solution = mds_solver_demds(G, seed=SEED, time_limit=TIME_LIMIT_SECONDS)
    except Exception as error:
        return {
            "status_python": "error",
            "time_python": perf_counter() - started,
            "error_python": f"{type(error).__name__} {error}",
        }
    elapsed = perf_counter() - started

    try:
        # Verifica a cobertura percorrendo as adjacências sem montar outro conjunto.
        valid = all(node in G for node in solution) and all(
            node in solution or any(neighbor in solution for neighbor in G[node])
            for node in G
        )
        return {
            "status_python": "ok" if valid else "invalid_solution",
            "output_python": len(solution),
            "time_python": elapsed,
            "python_valid": valid,
        }
    finally:
        del solution

def run_parameters() -> dict:
    if type(SEED) is not int or not -(1 << 31) <= SEED < (1 << 31):
        raise ValueError("SEED precisa ser um inteiro de 32 bits.")
    if type(TIME_LIMIT_SECONDS) is not int or not 0 <= TIME_LIMIT_SECONDS < (1 << 31):
        raise ValueError("TIME_LIMIT_SECONDS precisa ser um inteiro não negativo de 32 bits.")
    if type(CPP_ALPHA_ARG) is not int or not 0 <= CPP_ALPHA_ARG <= 100:
        raise ValueError("CPP_ALPHA_ARG precisa ser um inteiro entre 0 e 100.")
    if CPP_TIMEOUT_SECONDS is not None and CPP_TIMEOUT_SECONDS <= 0:
        raise ValueError("CPP_TIMEOUT_SECONDS precisa ser positivo.")

    executable = CPP_BIN.resolve()
    if not executable.is_file():
        raise FileNotFoundError(f"Ajuste CPP_BIN. Executável ausente em {executable}")

    graph_dir = GRAPH_DIR.resolve()
    return {
        "seed": SEED,
        "time_limit_s": TIME_LIMIT_SECONDS,
        "cpp_alpha_arg": CPP_ALPHA_ARG,
        "cpp_timeout_s": CPP_TIMEOUT_SECONDS,
        "cpp_bin": executable,
        "graph_dir": graph_dir,
    }

def compare_instance(path: Path, name: str, executable: Path, params: dict) -> dict:
    row = {"grafo": name, **params, "status_cpp": "skipped", "status_python": "skipped"}
    G = None
    try:
        try:
            G = load_edgelist(path)
            G = preprocess(G)
            row.update(n_nodes=G.number_of_nodes(), n_edges=G.number_of_edges())
        except Exception as error:
            row["error_input"] = f"{type(error).__name__} {error}"
            return row

        print(f"  C++ em {row['n_nodes']} vértices e {row['n_edges']} arestas", flush=True)
        try:
            row.update(run_cpp(G, executable, row["n_edges"]))
        except (OSError, ValueError, MemoryError) as error:
            row.update(status_cpp="error", error_cpp=f"{type(error).__name__} {error}")

        print("  Python", flush=True)
        row.update(run_python(G))

        if row["status_cpp"] == row["status_python"] == "ok":
            row["same_size"] = row["output_cpp"] == row["output_python"]
        return row
    finally:
        del G
        gc.collect()

## 3. Executar o lote e gravar o CSV

A célula grava e descarrega cada linha para o arquivo antes de avançar. Ao executar novamente com os mesmos parâmetros, ela pula os grafos já registrados, inclusive os que tiveram erro. Para mudar a semente, o limite ou o executável, escolha outro `CSV_PATH`.

In [6]:
CSV_COLUMNS = (
    "grafo", "output_cpp", "output_python", "time_python", "time_cpp",
    "seed", "time_limit_s", "cpp_alpha_arg", "cpp_timeout_s", "cpp_bin", "graph_dir",
    "n_nodes", "n_edges", "status_cpp", "status_python", "python_valid", "same_size",
    "error_input", "error_cpp", "error_python",
)


def read_completed(csv_path: Path, params: dict) -> set[str]:
    if not csv_path.exists() or csv_path.stat().st_size == 0:
        return set()

    expected = {key: "" if value is None else str(value) for key, value in params.items()}
    completed = set()
    with csv_path.open(encoding="utf-8", newline="") as stream:
        reader = csv.DictReader(stream, strict=True)
        if reader.fieldnames != list(CSV_COLUMNS):
            raise ValueError("O CSV existente tem outro cabeçalho. Escolha outro CSV_PATH.")
        for row in reader:
            if None in row or None in row.values() or not row["grafo"]:
                raise ValueError(f"Linha incompleta no CSV perto da linha {reader.line_num}.")
            if any(row[key] != value for key, value in expected.items()):
                raise ValueError("O CSV existente usa outros parâmetros. Escolha outro CSV_PATH.")
            if row["grafo"] in completed:
                raise ValueError(f"Grafo duplicado no CSV ({row['grafo']}).")
            completed.add(row["grafo"])
    return completed


params = run_parameters()
executable = params["cpp_bin"]
graph_dir = params["graph_dir"]
paths = list(iter_files(graph_dir, GRAPH_PATTERN, order=FILE_ORDER))
if not paths:
    raise FileNotFoundError(f"Nenhum arquivo em {graph_dir} para o padrão {GRAPH_PATTERN!r}.")

completed = read_completed(CSV_PATH, params)
CSV_PATH.parent.mkdir(parents=True, exist_ok=True)

written = 0
try:
    with CSV_PATH.open("a", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=CSV_COLUMNS)
        if stream.tell() == 0:
            writer.writeheader()
            stream.flush()

        for index, path in enumerate(paths, start=1):
            name = path.relative_to(graph_dir).as_posix().removesuffix(".txt.gz")
            if name in completed:
                continue

            print(f"[{index}/{len(paths)}] {name}", flush=True)
            row = compare_instance(path, name, executable, params)
            writer.writerow(row)
            stream.flush()
            completed.add(name)
            written += 1
            print(f"  Gravado | C++ {row['status_cpp']} | Python {row['status_python']}", flush=True)
except KeyboardInterrupt:
    print("Lote interrompido. As linhas concluídas já estão no CSV.")
    gc.collect()

print(f"{written} novas linhas em {CSV_PATH}")

[1/40] amazon0302
  C++ em 262111 vértices e 899792 arestas
  Python
  Gravado | C++ ok | Python ok
[2/40] amazon0312
  C++ em 400727 vértices e 2349869 arestas
  Python
  Gravado | C++ ok | Python ok
[3/40] amazon0505
  C++ em 410236 vértices e 2439437 arestas
  Python
  Gravado | C++ ok | Python ok
[4/40] amazon0601
  C++ em 403394 vértices e 2443408 arestas
  Python
  Gravado | C++ ok | Python ok
[5/40] as-skitter
  C++ em 1696415 vértices e 11095298 arestas
  Python
  Gravado | C++ ok | Python ok
[6/40] com-amazon
  C++ em 334863 vértices e 925872 arestas
  Python
  Gravado | C++ ok | Python ok
[7/40] com-dblp
  C++ em 317080 vértices e 1049866 arestas
  Python
  Gravado | C++ ok | Python ok
[8/40] com-youtube
  C++ em 1134890 vértices e 2987624 arestas
  Python
  Gravado | C++ ok | Python ok
[9/40] email-EuAll
  C++ em 265214 vértices e 364481 arestas
  Python
  Gravado | C++ ok | Python ok
[10/40] gemsec-Deezer-hr
  C++ em 54573 vértices e 498202 arestas
  Python
  Gravado | C++ 

In [15]:
instance_name = "amazon0302"

from pprint import pprint

TIME_LIMIT_SECONDS = 180

test_params = run_parameters()
instance_path = test_params["graph_dir"] / f"{instance_name}.txt.gz"
if not instance_path.is_file():
    raise FileNotFoundError(f"Instância ausente em {instance_path}")

resultado_teste = compare_instance(
    instance_path, instance_name, test_params["cpp_bin"], test_params
)
pprint(resultado_teste, sort_dicts=False)

  C++ em 262111 vértices e 899792 arestas
  Python
{'grafo': 'amazon0302',
 'seed': 1,
 'time_limit_s': 180,
 'cpp_alpha_arg': 90,
 'cpp_timeout_s': None,
 'cpp_bin': PosixPath('/home/fernando/complex-networks/libs/DemDS/DemDS'),
 'graph_dir': PosixPath('/home/fernando/complex-networks/data/raw'),
 'status_cpp': 'ok',
 'status_python': 'ok',
 'n_nodes': 262111,
 'n_edges': 899792,
 'output_cpp': 35619,
 'time_cpp': 181.3331704809998,
 'output_python': 35662,
 'time_python': 183.24812228600058,
 'python_valid': True,
 'same_size': False}


## 4. Conferir a tabela

As colunas de tempo estão em segundos. `same_size` é preenchida quando o C++ fornece uma cardinalidade e o conjunto Python é válido. As colunas de erro ajudam a identificar falhas de leitura ou execução.

In [11]:
import pandas as pd

resultados = pd.read_csv(
    CSV_PATH,
    dtype={"output_cpp": "Int64", "output_python": "Int64"},
)
resultados

,grafo,output_cpp,output_python,time_python,time_cpp,seed,time_limit_s,cpp_alpha_arg,cpp_timeout_s,cpp_bin,graph_dir,n_nodes,n_edges,status_cpp,status_python,python_valid,same_size,error_input,error_cpp,error_python
0,amazon0302,35664,35765,33.194602,31.339174,1,30,90,NaN,/home/fernando/complex-networks/libs/DemDS/DemDS,/home/fernando/complex-networks/data/raw,262111,899792,ok,ok,True,False,NaN,NaN,NaN
1,amazon0312,45567,45704,35.783452,31.730598,1,30,90,NaN,/home/fernando/complex-networks/libs/DemDS/DemDS,/home/fernando/complex-networks/data/raw,400727,2349869,ok,ok,True,False,NaN,NaN,NaN
2,amazon0505,47417,47549,36.016505,31.818615,1,30,90,NaN,/home/fernando/complex-networks/libs/DemDS/DemDS,/home/fernando/complex-networks/data/raw,410236,2439437,ok,ok,True,False,NaN,NaN,NaN
3,amazon0601,42385,42516,36.107439,31.808951,1,30,90,NaN,/home/fernando/complex-networks/libs/DemDS/DemDS,/home/fernando/complex-networks/data/raw,403394,2443408,ok,ok,True,False,NaN,NaN,NaN
4,as-skitter,182534,182627,43.296319,34.277633,1,30,90,NaN,/home/fernando/complex-networks/libs/DemDS/DemDS,/home/fernando/complex-networks/data/raw,1696415,11095298,ok,ok,True,False,NaN,NaN,NaN
5,com-amazon,52868,52893,33.270102,31.337182,1,30,90,NaN,/home/fernando/complex-networks/libs/DemDS/DemDS,/home/fernando/complex-networks/data/raw,334863,925872,ok,ok,True,False,NaN,NaN,NaN
6,com-dblp,46138,46138,32.569036,31.265305,1,30,90,NaN,/home/fernando/complex-networks/libs/DemDS/DemDS,/home/fernando/complex-networks/data/raw,317080,1049866,ok,ok,True,True,NaN,NaN,NaN
7,com-youtube,213122,213122,35.851847,31.833626,1,30,90,NaN,/home/fernando/complex-networks/libs/DemDS/DemDS,/home/fernando/complex-networks/data/raw,1134890,2987624,ok,ok,True,True,NaN,NaN,NaN
8,email-EuAll,18181,18181,31.751404,31.098739,1,30,90,NaN,/home/fernando/complex-networks/libs/DemDS/DemDS,/home/fernando/complex-networks/data/raw,265214,364481,ok,ok,True,True,NaN,NaN,NaN
9,gemsec-Deezer-hr,7110,7132,31.503813,31.084198,1,30,90,NaN,/home/fernando/complex-networks/libs/DemDS/DemDS,/home/fernando/complex-networks/data/raw,54573,498202,ok,ok,True,False,NaN,NaN,NaN


In [13]:
resultados[resultados['same_size']==False]

,grafo,output_cpp,output_python,time_python,time_cpp,seed,time_limit_s,cpp_alpha_arg,cpp_timeout_s,cpp_bin,graph_dir,n_nodes,n_edges,status_cpp,status_python,python_valid,same_size,error_input,error_cpp,error_python
0,amazon0302,35664,35765,33.194602,31.339174,1,30,90,NaN,/home/fernando/complex-networks/libs/DemDS/DemDS,/home/fernando/complex-networks/data/raw,262111,899792,ok,ok,True,False,NaN,NaN,NaN
1,amazon0312,45567,45704,35.783452,31.730598,1,30,90,NaN,/home/fernando/complex-networks/libs/DemDS/DemDS,/home/fernando/complex-networks/data/raw,400727,2349869,ok,ok,True,False,NaN,NaN,NaN
2,amazon0505,47417,47549,36.016505,31.818615,1,30,90,NaN,/home/fernando/complex-networks/libs/DemDS/DemDS,/home/fernando/complex-networks/data/raw,410236,2439437,ok,ok,True,False,NaN,NaN,NaN
3,amazon0601,42385,42516,36.107439,31.808951,1,30,90,NaN,/home/fernando/complex-networks/libs/DemDS/DemDS,/home/fernando/complex-networks/data/raw,403394,2443408,ok,ok,True,False,NaN,NaN,NaN
4,as-skitter,182534,182627,43.296319,34.277633,1,30,90,NaN,/home/fernando/complex-networks/libs/DemDS/DemDS,/home/fernando/complex-networks/data/raw,1696415,11095298,ok,ok,True,False,NaN,NaN,NaN
5,com-amazon,52868,52893,33.270102,31.337182,1,30,90,NaN,/home/fernando/complex-networks/libs/DemDS/DemDS,/home/fernando/complex-networks/data/raw,334863,925872,ok,ok,True,False,NaN,NaN,NaN
9,gemsec-Deezer-hr,7110,7132,31.503813,31.084198,1,30,90,NaN,/home/fernando/complex-networks/libs/DemDS/DemDS,/home/fernando/complex-networks/data/raw,54573,498202,ok,ok,True,False,NaN,NaN,NaN
10,gemsec-Deezer-hu,7746,7757,31.485250,31.042760,1,30,90,NaN,/home/fernando/complex-networks/libs/DemDS/DemDS,/home/fernando/complex-networks/data/raw,47538,222887,ok,ok,True,False,NaN,NaN,NaN
11,gemsec-Deezer-ro,9234,9237,31.259347,31.028424,1,30,90,NaN,/home/fernando/complex-networks/libs/DemDS/DemDS,/home/fernando/complex-networks/data/raw,41773,125826,ok,ok,True,False,NaN,NaN,NaN
12,gemsec-Facebook-artist,5467,5468,31.687653,31.128500,1,30,90,NaN,/home/fernando/complex-networks/libs/DemDS/DemDS,/home/fernando/complex-networks/data/raw,50515,819090,ok,ok,True,False,NaN,NaN,NaN
